Use test.csv from https://www.kaggle.com/datasets/kritanjalijain/amazon-reviews

Rename test.csv -> amazon_review.csv

In [1]:
import pandas as pd
import re
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

# Load the dataset
# Assuming 'amazon_reviews.csv' with columns 'reviewText' and 'sentiment'
data = pd.read_csv('amazon_reviews.csv')

# Preprocess the text
stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    # Lowercase, remove punctuation and stop words
    text = re.sub(r'[^\w\s]', '', text.lower())
    tokens = word_tokenize(text)
    return ' '.join([word for word in tokens if word not in stop_words])

# Apply preprocessing
data['cleaned_review'] = data['reviewText'].apply(preprocess_text)
data.head()


,polarity,reviewTextSummary,reviewText,cleaned_review
0,2,Great CD,My lovely Pat has one of the GREAT voices of h...,lovely pat one great voices generation listene...
1,2,One of the best game music soundtracks - for a...,Despite the fact that I have only played a sma...,despite fact played small portion game music h...
2,1,Batteries died within a year ...,I bought this charger in Jul 2003 and it worke...,bought charger jul 2003 worked ok design nice ...
3,2,"works fine, but Maha Energy is better",Check out Maha Energy's website. Their Powerex...,check maha energys website powerex mhc204f cha...
4,2,Great for the non-audiophile,Reviewed quite a bit of the combo players and ...,reviewed quite bit combo players hesitant due ...


In [2]:
import gensim.downloader as api
import numpy as np

# Load pre-trained Word2Vec model (Google News, for example)
w2v_model = api.load("word2vec-google-news-300")

# Function to convert review text to an embedding
def get_review_embedding(text):
    tokens = text.split()
    word_vectors = [w2v_model[word] for word in tokens if word in w2v_model]
    if len(word_vectors) > 0:
        return np.mean(word_vectors, axis=0)
    else:
        return np.zeros(300)  # If no words match in the model

# Generate embeddings for each review
data['embedding'] = data['cleaned_review'].apply(get_review_embedding)
X = np.vstack(data['embedding'].values)
y = data['polarity'].values
data.head()

,polarity,reviewTextSummary,reviewText,cleaned_review,embedding
0,2,Great CD,My lovely Pat has one of the GREAT voices of h...,lovely pat one great voices generation listene...,"[0.058009513, 0.046717558, 0.023809537, 0.0724..."
1,2,One of the best game music soundtracks - for a...,Despite the fact that I have only played a sma...,despite fact played small portion game music h...,"[0.09045663, 0.011980617, 0.020219915, 0.08089..."
2,1,Batteries died within a year ...,I bought this charger in Jul 2003 and it worke...,bought charger jul 2003 worked ok design nice ...,"[-0.0011805781, 0.052590545, 0.015240139, 0.04..."
3,2,"works fine, but Maha Energy is better",Check out Maha Energy's website. Their Powerex...,check maha energys website powerex mhc204f cha...,"[-0.006930542, 0.04350179, 0.088702396, 0.0232..."
4,2,Great for the non-audiophile,Reviewed quite a bit of the combo players and ...,reviewed quite bit combo players hesitant due ...,"[0.0283665, 0.083570994, -0.029844955, 0.12375..."


In [3]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train a Logistic Regression model
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

# Make predictions on the test set
y_pred = model.predict(X_test)

# Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
print("Accuracy:", accuracy)
print("F1 Score:", f1)
print("Classification Report:\n", classification_report(y_test, y_pred))


Accuracy: 0.8153375
F1 Score: 0.8161121276622229
Classification Report:
               precision    recall  f1-score   support

           1       0.81      0.82      0.82     39896
           2       0.82      0.81      0.81     40104

    accuracy                           0.82     80000
   macro avg       0.82      0.82      0.82     80000
weighted avg       0.82      0.82      0.82     80000



In [4]:
data['reviewText'].loc[0]

'My lovely Pat has one of the GREAT voices of her generation. I have listened to this CD for YEARS and I still LOVE IT. When I\'m in a good mood it makes me feel better. A bad mood just evaporates like sugar in the rain. This CD just oozes LIFE. Vocals are jusat STUUNNING and lyrics just kill. One of life\'s hidden gems. This is a desert isle CD in my book. Why she never made it big is just beyond me. Everytime I play this, no matter black, white, young, old, male, female EVERYBODY says one thing "Who was that singing ?"'

In [16]:
testvec = data['embedding'].loc[0]
testvec

array([ 5.80095127e-02,  4.67175581e-02,  2.38095373e-02,  7.24631548e-02,
       -4.10527848e-02, -4.40757070e-03,  2.78897714e-02, -1.01261072e-01,
        8.35937485e-02,  1.07738703e-01, -2.23965738e-02, -1.14985794e-01,
       -3.41131026e-03,  2.74597164e-02, -8.87462273e-02,  7.79811516e-02,
        7.49991685e-02,  1.38283476e-01, -2.74439026e-02, -9.86333638e-02,
       -1.44032994e-02,  7.42507949e-02,  2.86476836e-02, -2.76786108e-02,
       -1.16438428e-02, -1.04265390e-02, -1.22228310e-01,  4.93719615e-02,
        1.34338383e-02, -2.42009945e-02, -5.65399155e-02,  3.96459401e-02,
       -3.74425724e-02,  4.90489602e-02, -1.21909399e-02, -2.04053018e-02,
        3.33093815e-02,  1.06614549e-02,  2.84171365e-02,  8.28033462e-02,
        9.12064984e-02, -5.11783957e-02,  1.14075817e-01,  1.17109818e-02,
       -2.97407678e-04, -5.94510175e-02, -2.57252082e-02, -2.02203933e-02,
       -1.60250571e-02,  2.26085316e-02, -4.58767973e-02,  6.93814382e-02,
       -1.01595791e-03, -

In [7]:
model.predict([["This is an amazing product!"]])

ValueError: dtype='numeric' is not compatible with arrays of bytes/strings.Convert your data to numeric values explicitly instead.

In [17]:
model.predict([testvec])[0]

np.int64(2)

In [18]:
data.head()

,polarity,reviewTextSummary,reviewText,cleaned_review,embedding
0,2,Great CD,My lovely Pat has one of the GREAT voices of h...,lovely pat one great voices generation listene...,"[0.058009513, 0.046717558, 0.023809537, 0.0724..."
1,2,One of the best game music soundtracks - for a...,Despite the fact that I have only played a sma...,despite fact played small portion game music h...,"[0.09045663, 0.011980617, 0.020219915, 0.08089..."
2,1,Batteries died within a year ...,I bought this charger in Jul 2003 and it worke...,bought charger jul 2003 worked ok design nice ...,"[-0.0011805781, 0.052590545, 0.015240139, 0.04..."
3,2,"works fine, but Maha Energy is better",Check out Maha Energy's website. Their Powerex...,check maha energys website powerex mhc204f cha...,"[-0.006930542, 0.04350179, 0.088702396, 0.0232..."
4,2,Great for the non-audiophile,Reviewed quite a bit of the combo players and ...,reviewed quite bit combo players hesitant due ...,"[0.0283665, 0.083570994, -0.029844955, 0.12375..."
